# 05 — Error analysis (proposal 3.3.5) → RQ3

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

Confusion matrix → pairwise confusion scores → cluster rows → within- vs cross-family comparison.

In [ ]:
# Self-contained rebuild of the best model (exp6_all + logreg, same as nb04)
# so this notebook has no hidden dependency on 03/04 having run first.
from src import data, features, modeling, analysis, viz

clean = data.load_or_build_clean(); splits = data.load_or_build_splits(clean)
y = clean[config.LABEL_COL].values
texts = clean['text']

X_tfidf, _ = features.build_tfidf(texts.iloc[splits['train']], texts)
sty = features.build_stylometric(texts)
sty_scaled, _ = features.scale_dense(sty.values, splits['train'])
bib = features.build_biber(texts)
bib_scaled, _ = features.scale_dense(bib.values, splits['train'])
emb = features.build_sbert(texts)   # cached

blocks = {'tfidf': X_tfidf, 'stylometric': sty_scaled, 'biber': bib_scaled, 'sbert': emb,
          'length': clean[['log_token_count']].values}
X = features.assemble(blocks, features.EXPERIMENTS['exp6_all'])
Xtr, Xval = X[splits['train']], X[splits['val']]
ytr, yval = y[splits['train']], y[splits['val']]

res = modeling.train_and_evaluate('logreg', Xtr, ytr, Xval, yval, labels=list(config.CLASSES))
conf, labels = res.confusion, res.labels
print('Macro-F1 (exp6_all/logreg):', res.macro_f1)

In [ ]:
conf_arr = np.asarray(conf, dtype=float)
conf_norm = conf_arr / conf_arr.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7), facecolor=viz.SURFACE)
im = ax.imshow(conf_norm, cmap=viz.SEQUENTIAL_BLUE, vmin=0, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix (row-normalized) — Exp 6 (all features) + logreg')
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Share of true-class rows')
fig.tight_layout()
save_fig(fig, 'confusion_matrix_heatmap')

In [ ]:
pw = analysis.pairwise_confusion_scores(conf, labels)
summary = analysis.within_vs_cross_family(pw)
print(summary)
pw.head(10)

In [ ]:
fig, ax = viz.new_fig(figsize=(4.5, 5))
cats = ['Within-family', 'Cross-family']
vals = [summary['within_family_mean'], summary['cross_family_mean']]
bars = ax.bar(cats, vals, color=viz.CATEGORICAL[:2], width=0.55, zorder=3)
ax.bar_label(bars, fmt='%.3f', color=viz.INK_SECONDARY, padding=3)
ax.set_ylabel('Mean pairwise confusion score')
ax.set_title(f"Within- vs cross-family confusion\n(n={summary['within_family_n']} vs {summary['cross_family_n']} pairs)")
fig.tight_layout()
save_fig(fig, 'within_vs_cross_family_confusion')

In [ ]:
from scipy.cluster.hierarchy import dendrogram

link = analysis.cluster_confusion_rows(conf, labels)
fig, ax = plt.subplots(figsize=(9, 5), facecolor=viz.SURFACE)
dendrogram(link, labels=labels, ax=ax, color_threshold=0, above_threshold_color=viz.INK_SECONDARY)
ax.set_title('Hierarchical clustering of confusion rows (Ward linkage)\ncompare groupings against config.MODEL_FAMILIES')
ax.set_ylabel('Distance')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelrotation=45, labelsize=8, colors=viz.INK_SECONDARY)
ax.tick_params(axis='y', colors=viz.INK_SECONDARY)
fig.tight_layout()
save_fig(fig, 'confusion_dendrogram')